# OVITO Integration with Volt

This notebook demonstrates how to use **OVITO** with trajectory data stored in Volt.
You will learn how to:
1. Connect to Volt and load a trajectory
2. Download dump frames from the platform
3. Create an OVITO pipeline from Volt data
4. Apply analysis modifiers (CNA, PTM, Atomic Strain, etc.)
5. Track properties across frames
6. Plot results with matplotlib

## 1. Connect to Volt

Inside Volt's Jupyter environment, authentication is automatic via environment variables.
If you are running this externally, provide your secret key directly.

In [ ]:
from voltsdk import VoltClient
import os

# Inside Volt Jupyter: credentials are auto-detected
client = VoltClient.from_env()

# Alternatively, provide credentials explicitly:
# client = VoltClient(
#     secret_key="vsk_your_key_here",
#     base_url="https://server.voltcloud.dev/api"
# )

print(f"Connected to team: {client.team.name}")

## 2. Load a Trajectory

If this notebook was opened from a trajectory in Volt, the trajectory ID is
available as an environment variable. Otherwise, you can browse your trajectories.

In [ ]:
# Load trajectory from environment (if opened from Volt UI)
trajectory_id = os.environ.get('VOLT_TRAJECTORY_ID', '')

if trajectory_id:
    traj = client.trajectories.get(trajectory_id)
else:
    # Browse available trajectories and pick the first one
    print("No trajectory linked. Listing available trajectories:")
    for t in client.trajectories.list():
        print(f"  [{t.id}] {t.name} ({t.status}, {t.frame_count} frames)")
    traj = client.trajectories.first()

if traj:
    print(f"\nTrajectory: {traj.name}")
    print(f"Status: {traj.status}")
    print(f"Frames: {traj.frame_count}")
else:
    print("No trajectories found. Upload a simulation first.")

## 3. Download Dump Frames

OVITO works with local dump files. The SDK downloads frames from Volt's
storage and decompresses them automatically.

You can download all frames or select specific timesteps.

In [ ]:
import gzip
import shutil

# Select which frames to download
# Option A: First and last frame only (quick test)
frames_to_download = [traj.frames[0], traj.frames[-1]]

# Option B: Every 10th frame
# frames_to_download = [f for i, f in enumerate(traj.frames) if i % 10 == 0]

# Option C: All frames
# frames_to_download = list(traj.frames)

dump_dir = f'./volt_dumps/{traj.id}/'
os.makedirs(dump_dir, exist_ok=True)

dump_files = []
for frame in frames_to_download:
    # Download the compressed dump file
    gz_path = frame.download_dump(dest=dump_dir)
    
    # Decompress for OVITO compatibility
    if gz_path.endswith('.gz'):
        dump_path = gz_path[:-3]
        with gzip.open(gz_path, 'rb') as f_in:
            with open(dump_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        dump_files.append(dump_path)
    else:
        dump_files.append(gz_path)
    
    print(f"  Downloaded: timestep {frame.timestep} ({frame.natoms} atoms)")

print(f"\nTotal files: {len(dump_files)}")

## 4. Create an OVITO Pipeline

Load the downloaded dump files into an OVITO pipeline.
This gives you full access to OVITO's analysis and visualization capabilities.

In [ ]:
from ovito.io import import_file

# Import the first dump file (or all files for multi-frame analysis)
if len(dump_files) == 1:
    pipeline = import_file(dump_files[0])
else:
    pipeline = import_file(dump_files[0])
    # Note: For a full time series, ensure files follow sequential naming
    # or import them as a file sequence

# Inspect the data
data = pipeline.compute()
print(f"Number of atoms: {data.particles.count}")
print(f"Particle types: {set(data.particles['Particle Type'].array)}")
print(f"Simulation cell:")
print(f"  {data.cell}")

# List available particle properties
print(f"\nAvailable properties:")
for prop in data.particles.keys():
    print(f"  - {prop}")

## 5. Structural Analysis with CNA

Apply the **Common Neighbor Analysis (CNA)** modifier to identify
crystal structures (FCC, BCC, HCP, etc.) in your simulation.

In [ ]:
from ovito.modifiers import CommonNeighborAnalysisModifier

# Add CNA modifier
cna = CommonNeighborAnalysisModifier()
pipeline.modifiers.append(cna)

# Compute
data = pipeline.compute()

# Get structure type counts
struct_types = data.particles['Structure Type'].array

# CNA structure type mapping:
# 0 = Other, 1 = FCC, 2 = HCP, 3 = BCC, 4 = ICO
structure_names = {0: 'Other', 1: 'FCC', 2: 'HCP', 3: 'BCC', 4: 'ICO'}

print("Structure Analysis Results:")
total = len(struct_types)
for type_id, name in structure_names.items():
    count = (struct_types == type_id).sum()
    pct = count / total * 100
    print(f"  {name}: {count} atoms ({pct:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the structure distribution
labels = []
counts = []
colors = ['#999999', '#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for type_id, name in structure_names.items():
    count = (struct_types == type_id).sum()
    if count > 0:
        labels.append(name)
        counts.append(count)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, counts, color=colors[:len(labels)])
ax.set_ylabel('Number of Atoms')
ax.set_title('Crystal Structure Distribution (CNA)')

# Add count labels on bars
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{count}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Atomic Strain Analysis

If you have multiple frames, you can compute atomic strain relative
to a reference configuration (typically the first frame).

In [ ]:
from ovito.modifiers import AtomicStrainModifier

# Only useful if we have multiple frames
if len(dump_files) > 1:
    # Create a fresh pipeline with all frames
    pipeline_strain = import_file(dump_files)
    
    # Add atomic strain modifier (uses first frame as reference)
    strain_mod = AtomicStrainModifier(
        cutoff=3.5,  # Adjust based on your material's lattice constant
        output_strain_tensors=True
    )
    pipeline_strain.modifiers.append(strain_mod)
    
    # Compute on the last frame
    last_frame = pipeline_strain.source.num_frames - 1
    data_strain = pipeline_strain.compute(last_frame)
    
    shear_strain = data_strain.particles['Shear Strain'].array
    
    print(f"Shear strain statistics (frame {last_frame}):")
    print(f"  Mean: {np.mean(shear_strain):.4f}")
    print(f"  Max:  {np.max(shear_strain):.4f}")
    print(f"  Std:  {np.std(shear_strain):.4f}")
    
    # Plot strain distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(shear_strain, bins=100, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Shear Strain')
    ax.set_ylabel('Number of Atoms')
    ax.set_title('Shear Strain Distribution')
    ax.axvline(np.mean(shear_strain), color='red', linestyle='--', label=f'Mean: {np.mean(shear_strain):.4f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Atomic strain requires at least 2 frames.")
    print("Download more frames in Section 3 to use this analysis.")

## 7. Per-Frame Analysis: Tracking Properties Over Time

Run an OVITO modifier across all frames to track how a property
evolves during the simulation.

In [ ]:
# Download more frames for time evolution analysis
# (Skip if you already downloaded all frames above)

# For this example, download every Nth frame
N = max(1, traj.frame_count // 20)  # ~20 frames total
selected_frames = [f for i, f in enumerate(traj.frames) if i % N == 0]

all_dump_files = []
for frame in selected_frames:
    gz_path = frame.download_dump(dest=dump_dir)
    if gz_path.endswith('.gz'):
        dump_path = gz_path[:-3]
        if not os.path.exists(dump_path):
            with gzip.open(gz_path, 'rb') as f_in:
                with open(dump_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
        all_dump_files.append(dump_path)
    else:
        all_dump_files.append(gz_path)

print(f"Downloaded {len(all_dump_files)} frames for time evolution analysis")

In [ ]:
# Track CNA structure fractions across frames

pipeline_evolution = import_file(all_dump_files[0])
pipeline_evolution.modifiers.append(CommonNeighborAnalysisModifier())

timesteps = []
fcc_fractions = []
hcp_fractions = []
other_fractions = []

for i, dump_file in enumerate(all_dump_files):
    # Load each file individually
    p = import_file(dump_file)
    p.modifiers.append(CommonNeighborAnalysisModifier())
    data = p.compute()
    
    struct = data.particles['Structure Type'].array
    total = len(struct)
    
    timesteps.append(selected_frames[i].timestep)
    fcc_fractions.append((struct == 1).sum() / total * 100)
    hcp_fractions.append((struct == 2).sum() / total * 100)
    other_fractions.append((struct == 0).sum() / total * 100)

print(f"Analyzed {len(timesteps)} frames")

In [ ]:
# Plot structure evolution over time
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(timesteps, fcc_fractions, 'o-', label='FCC', color='#2196F3', linewidth=2)
ax.plot(timesteps, hcp_fractions, 's-', label='HCP', color='#4CAF50', linewidth=2)
ax.plot(timesteps, other_fractions, '^-', label='Other/Disordered', color='#999999', linewidth=2)

ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('Fraction (%)', fontsize=12)
ax.set_title('Crystal Structure Evolution', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Combining Volt Listings with OVITO Results

You can combine data from Volt's analysis listings (e.g., thermodynamic
properties computed by LAMMPS) with structural analysis from OVITO.

In [ ]:
import pandas as pd

# Get listing data from Volt (if analyses exist)
volt_df = None
analysis = traj.analyses.first()
if analysis:
    volt_df = analysis.listings.to_dataframe()
    print(f"Volt listing columns: {volt_df.columns.tolist()}")
    print(volt_df.head())
else:
    print("No analyses found for this trajectory.")
    print("Run an analysis plugin in Volt to generate listing data.")

In [ ]:
# Create a combined DataFrame with OVITO results
ovito_df = pd.DataFrame({
    'timestep': timesteps,
    'fcc_fraction': fcc_fractions,
    'hcp_fraction': hcp_fractions,
    'disordered_fraction': other_fractions
})

# If we have Volt listing data, merge on timestep
if volt_df is not None and 'timestep' in volt_df.columns:
    combined = pd.merge(ovito_df, volt_df, on='timestep', how='inner')
    print(f"Combined DataFrame: {len(combined)} rows, {len(combined.columns)} columns")
    combined.head()
else:
    print("OVITO analysis results:")
    ovito_df.head()

## 9. Export Processed Data

Save your analysis results for further processing or publication.

In [ ]:
# Export OVITO results to CSV
ovito_df.to_csv('ovito_structure_analysis.csv', index=False)
print("Saved: ovito_structure_analysis.csv")

# Export processed atoms from OVITO (with structure types)
from ovito.io import export_file

if len(all_dump_files) > 0:
    p = import_file(all_dump_files[-1])
    p.modifiers.append(CommonNeighborAnalysisModifier())
    
    export_file(p, 'processed_atoms.dump', 'lammps/dump',
                columns=['Particle Identifier', 'Particle Type',
                         'Position.X', 'Position.Y', 'Position.Z',
                         'Structure Type'])
    print("Saved: processed_atoms.dump")

## Quick Reference

| Task | Code |
|------|------|
| Connect to Volt | `client = VoltClient.from_env()` |
| Get trajectory | `traj = client.trajectories.get("id")` |
| Download dump | `frame.download_dump(dest="./dumps/")` |
| OVITO pipeline (shortcut) | `pipeline = traj.to_ovito_pipeline()` |
| Single frame to OVITO | `data = frame.to_ovito_data()` |
| CNA modifier | `pipeline.modifiers.append(CommonNeighborAnalysisModifier())` |
| PTM modifier | `pipeline.modifiers.append(PolyhedralTemplateMatchingModifier())` |
| Atomic strain | `pipeline.modifiers.append(AtomicStrainModifier(cutoff=3.5))` |
| Export dump | `export_file(pipeline, 'out.dump', 'lammps/dump', columns=[...])` |
| Listings to DataFrame | `df = analysis.listings.to_dataframe()` |